# Phase 5: Verifiable-Reward RL (GRPO on Strict Trace Failures)

Evaluates the Phase 5 **GRPO Reinforcement Learning** checkpoint trained strictly on filtered hard `train` failures using the binary `{0.0, 1.0}` strict-trace verifier:

1. Strict regex format (`^Work:\s*([^\r\n]+)\r?\nAnswer:\s*(-?\d+)$`) + clean `<|eos|>` stop.
2. Every `;`-separated `Work:` equation holds (`LHS == RHS`, exact division, no operator-free tautologies).
3. Final `Work:` step equals `Answer:`.
4. `Answer:` equals ground truth.

- **Mine Failed Hard Dataset:** `bash scripts/prepare_data.sh --stage rl`
- **Launch GRPO Training:** `bash scripts/train.sh grpo --preset grpo`

In [ ]:
from pathlib import Path
import sys

root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "main.py").is_file()), Path.cwd().parent)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.infra.eval_setup import setup_notebook_environment, configure_eval_device

project_root = setup_notebook_environment(root)
device, dtype = configure_eval_device("auto")  # 'auto', 'cuda', or 'cpu'

## 1. Load Phase 5 GRPO Checkpoint & Frozen CoT SFT Parent

In [ ]:
from src.infra.eval_setup import select_run_and_checkpoint, load_checkpoint_bundle, plot_run_history

RUN_ID = None                # Auto-selects newest active Phase 5 RL/GRPO run
CHECKPOINT_ALIAS = "best"    # 'best' selects lowest (1 - val_strict_pass); or 'latest' / 'step-N'
LOCAL_CHECKPOINT = None
COMPARE_WITH_PARENT_SFT = True

run_id, ckpt_path, snapshot = select_run_and_checkpoint(
    run_id=RUN_ID,
    checkpoint_alias=CHECKPOINT_ALIAS,
    local_checkpoint=LOCAL_CHECKPOINT,
    preferred_stages=("rl", "grpo"),
)
bundle = load_checkpoint_bundle(ckpt_path, device, dtype, snapshot=snapshot, compare_with_base=COMPARE_WITH_PARENT_SFT)
model, tokenizer, model_config = bundle["model"], bundle["tokenizer"], bundle["model_config"]
arith_manifest = bundle["arith_manifest"]
plot_run_history(bundle["metrics_history"], run_id=run_id, model_config=model_config, stage=bundle["stage"], snapshot=snapshot)

## 2. Strict-Trace Evaluation on Held-Out `val` (`RL` vs. Parent CoT `SFT`)

In [ ]:
from src.core.benchmark import evaluate_arithmetic_split

EVAL_SPLIT = "val"         # Keep 'val' until Phase 6 sealed release check
MAX_EXAMPLES = None if device.type == "cuda" else 280
BATCH_SIZE = 64 if device.type == "cuda" else 32

grpo_val_res = evaluate_arithmetic_split(
    model,
    tokenizer,
    model_config,
    arith_manifest,
    device=device,
    dtype=dtype,
    eval_split=EVAL_SPLIT,
    max_examples=MAX_EXAMPLES,
    batch_size=BATCH_SIZE,
    use_kv_cache=True,
    max_misses_to_show=15,
    base_model=bundle["base_model"],
    base_tokenizer=bundle["base_tokenizer"],
    strict_trace=True,
)

## 3. Hard-Case Stress Check & Story/Math Prompt Playground

In [ ]:
from src.core.benchmark import create_ask_helper, create_math_chat_helpers

ask_math, _ = create_math_chat_helpers(
    model, tokenizer, device=device, answer_format="scratchpad", use_cache=True,
)
ask = create_ask_helper(
    model, tokenizer, device=device, answer_format="scratchpad", use_cache=True,
)

hard_probes = [
    ("What is 177 * 515?", 91155),
    ("What is -893 / (-19)?", 47),
    ("What is -875 / (-25)?", 35),
    ("What is -765 / 15?", -51),
    ("What is -95 * 672?", -63840),
    ("Compute 15 * 4 and 92 - 17, then subtract the second result from the first.", -15),
]
for q, gold in hard_probes:
    ask_math(q, expected=gold)

In [ ]:
_ = ask("What is 177 * 515?")